<a href="https://colab.research.google.com/github/rachmadnanda/MachineLearning2026/blob/main/jobsheet_pbl/JS11_PBL_244107020095_RACHMAD_FEBRIANANDA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# JS11 — Klasifikasi dengan Support Vector Machine (SVM)

Jobsheet asli memakai data dummy, citra wajah LFW, dan klasifikasi siang/malam.
Versi PBL ini memakai SVM untuk memisahkan **Murah / Sedang / Mahal** pada data
properti, sekaligus mengilustrasikan margin dan pengaruh parameter `C` serta `gamma`.

SVM mencari bidang pemisah dengan **margin terbesar**. Untuk data yang tidak linear,
kernel RBF memetakan data ke dimensi lebih tinggi agar bisa dipisahkan.

## Langkah 1 — Muat Data dan Bagi Train/Test

Fitur yang dipakai: empat atribut fisik + frekuensi lokasi. Standardisasi dilakukan di
dalam pipeline karena SVM sangat sensitif terhadap skala fitur.

In [1]:
import os, warnings
warnings.filterwarnings('ignore')
os.environ.setdefault('TF_CPP_MIN_LOG_LEVEL', '3')
os.environ.setdefault('TF_ENABLE_ONEDNN_OPTS', '0')

import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.svm import SVC, LinearSVC
from sklearn.decomposition import PCA
from sklearn.metrics import (accuracy_score, classification_report,
                             confusion_matrix, ConfusionMatrixDisplay)

pd.set_option('display.width', 120)
sns.set_theme(style='whitegrid', palette='muted')
RANDOM_STATE = 42

df = pd.read_csv('dataset/properti_multi_kota.csv').drop_duplicates()
df = df[(df['luas_m2'] >= 10) & (df['luas_m2'] <= 2000)]
df = df[df['luas_bangunan'].isna() |
        ((df['luas_bangunan'] >= 10) & (df['luas_bangunan'] <= 2000))]
df = df[df['harga'].between(50_000_000, 100_000_000_000)]
df = df[df['jumlah_kamar'].isna() | (df['jumlah_kamar'] <= 20)]
df = df[df['jumlah_kamar_mandi'].isna() | (df['jumlah_kamar_mandi'] <= 20)]

FITUR_NUM = ['luas_m2', 'luas_bangunan', 'jumlah_kamar', 'jumlah_kamar_mandi']
df[FITUR_NUM] = df[FITUR_NUM].fillna(df[FITUR_NUM].median())
df['lokasi_bersih'] = df['lokasi'].str.strip().str.title()
df['lokasi_freq'] = df['lokasi_bersih'].map(df['lokasi_bersih'].value_counts(normalize=True))

q1, q2 = df['harga'].quantile([1 / 3, 2 / 3])
df['kelas_harga'] = np.where(df['harga'] < q1, 'Murah',
                     np.where(df['harga'] <= q2, 'Sedang', 'Mahal'))

FITUR = FITUR_NUM + ['lokasi_freq']
X = df[FITUR]
y = df['kelas_harga']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=RANDOM_STATE, stratify=y)

print(f'Train: {X_train.shape[0]:,} | Test: {X_test.shape[0]:,}')

Train: 39,319 | Test: 16,851


## Langkah 2 — Ilustrasi Batas Keputusan (Linear vs RBF)

Agar bisa digambar, dipakai dua fitur (luas tanah dan luas bangunan) dan dua kelas saja
(Murah vs Mahal) pada sampel 1.500 baris. Bidang berwarna menunjukkan wilayah prediksi;
garis adalah batas keputusan.

In [2]:
biner = df[df['kelas_harga'].isin(['Murah', 'Mahal'])].sample(1500, random_state=RANDOM_STATE)
Xb = StandardScaler().fit_transform(biner[['luas_m2', 'luas_bangunan']])
yb = (biner['kelas_harga'] == 'Mahal').astype(int).values

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
xx, yy = np.meshgrid(np.linspace(Xb[:, 0].min(), Xb[:, 0].max(), 200),
                     np.linspace(Xb[:, 1].min(), Xb[:, 1].max(), 200))

for ax, kernel in zip(axes, ['linear', 'rbf']):
    svm = SVC(kernel=kernel, C=1.0, gamma='scale').fit(Xb, yb)
    Z = svm.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
    ax.contourf(xx, yy, Z, alpha=0.25, cmap='coolwarm')
    ax.scatter(Xb[:, 0], Xb[:, 1], c=yb, cmap='coolwarm', s=12, edgecolors='k', linewidths=0.2)
    ax.set_title(f'SVM kernel {kernel}')
    ax.set_xlabel('luas_m2 (standar)')
    ax.set_ylabel('luas_bangunan (standar)')

plt.tight_layout()
plt.show()

## Langkah 3 — Perbandingan Kernel

Tiga kernel diuji pada sampel 12.000 baris latih dan dievaluasi pada seluruh data uji:

- **Linear** — cocok jika kelas hampir terpisah linear.
- **RBF** — memetakan data ke dimensi tak hingga; paling fleksibel.
- **Polinomial derajat 3** — menangkap interaksi antarfitur.

In [3]:
idx = np.random.default_rng(RANDOM_STATE).choice(len(X_train), 12_000, replace=False)
X_sub, y_sub = X_train.iloc[idx], y_train.iloc[idx]

hasil = []

def uji_kernel(nama, model):
    t0 = time.perf_counter()
    model.fit(X_sub, y_sub)
    waktu = time.perf_counter() - t0
    akurasi = accuracy_score(y_test, model.predict(X_test))
    hasil.append({'Model': nama, 'Akurasi': akurasi, 'Waktu fit (s)': waktu})
    return model

uji_kernel('LinearSVC', make_pipeline(StandardScaler(), LinearSVC(max_iter=5000)))
uji_kernel('SVC RBF', make_pipeline(StandardScaler(), SVC(kernel='rbf', C=1.0)))
uji_kernel('SVC Polinomial', make_pipeline(StandardScaler(), SVC(kernel='poly', degree=3, C=1.0)))

pd.DataFrame(hasil).round(4)

,Model,Akurasi,Waktu fit (s)
0,LinearSVC,0.6540,0.0859
1,SVC RBF,0.7155,2.1949
2,SVC Polinomial,0.6628,5.1738


## Langkah 4 — Pengaruh Parameter `C`

`C` mengatur trade-off antara margin lebar dan kesalahan klasifikasi:

- `C` kecil → margin lebar, lebih toleran terhadap kesalahan (bias tinggi).
- `C` besar → margin sempit, berusaha benar pada semua data latih (variansi tinggi).

Eksperimen memakai 6.000 baris agar cepat.

In [4]:
idx_kecil = np.random.default_rng(RANDOM_STATE).choice(len(X_train), 6_000, replace=False)
X_kecil, y_kecil = X_train.iloc[idx_kecil], y_train.iloc[idx_kecil]

baris_c = []
for C in [0.01, 0.1, 1, 10, 100]:
    model = make_pipeline(StandardScaler(), SVC(kernel='rbf', C=C, gamma='scale'))
    model.fit(X_kecil, y_kecil)
    baris_c.append({'C': C, 'Akurasi test': accuracy_score(y_test, model.predict(X_test))})

tabel_c = pd.DataFrame(baris_c)
print(tabel_c.round(4).to_string(index=False))

plt.figure(figsize=(7, 4))
plt.semilogx(tabel_c['C'], tabel_c['Akurasi test'], 'o-')
plt.xlabel('C (skala log)')
plt.ylabel('Akurasi test')
plt.title('Pengaruh C pada SVM RBF')
plt.tight_layout()
plt.show()

     C  Akurasi test
  0.01        0.6620
  0.10        0.6871
  1.00        0.7124
 10.00        0.7155
100.00        0.7152


## Langkah 5 — Pencarian Hyperparameter (GridSearchCV)

Grid kecil (`C` × `gamma`) diuji dengan 3-fold CV pada 6.000 baris. Setelah parameter
terbaik ditemukan, model dilatih ulang pada 20.000 baris dan diuji pada seluruh test set.

In [5]:
grid = {
    'svc__C': [1, 10],
    'svc__gamma': ['scale', 0.1],
}
pencarian = GridSearchCV(
    make_pipeline(StandardScaler(), SVC(kernel='rbf')),
    grid, cv=StratifiedKFold(3, shuffle=True, random_state=RANDOM_STATE),
    scoring='accuracy', n_jobs=-1)
t0 = time.perf_counter()
pencarian.fit(X_kecil, y_kecil)
print(f'Waktu GridSearch: {time.perf_counter() - t0:.1f} detik')
print(f'Parameter terbaik: {pencarian.best_params_}')
print(f'Akurasi CV terbaik: {pencarian.best_score_:.4f}')

Waktu GridSearch: 5.9 detik
Parameter terbaik: {'svc__C': 10, 'svc__gamma': 'scale'}
Akurasi CV terbaik: 0.7233


In [6]:
idx_final = np.random.default_rng(7).choice(len(X_train), 20_000, replace=False)
X_final, y_final = X_train.iloc[idx_final], y_train.iloc[idx_final]

svm_final = make_pipeline(
    StandardScaler(),
    SVC(kernel='rbf', C=pencarian.best_params_['svc__C'],
        gamma=pencarian.best_params_['svc__gamma']))
t0 = time.perf_counter()
svm_final.fit(X_final, y_final)
waktu = time.perf_counter() - t0
pred = svm_final.predict(X_test)

print(f'Waktu fit (20k) : {waktu:.1f} detik')
print(f'Akurasi test    : {accuracy_score(y_test, pred):.4f}')
print(f'Jumlah support vector: {svm_final[-1].n_support_.sum():,}')
print()
print(classification_report(y_test, pred, digits=4))

Waktu fit (20k) : 6.6 detik
Akurasi test    : 0.7204
Jumlah support vector: 12,088

              precision    recall  f1-score   support

       Mahal     0.8108    0.7821    0.7962      5616
       Murah     0.7399    0.8113    0.7739      5617
      Sedang     0.6049    0.5680    0.5859      5618

    accuracy                         0.7204     16851
   macro avg     0.7185    0.7204    0.7187     16851
weighted avg     0.7185    0.7204    0.7186     16851



In [7]:
cm = confusion_matrix(y_test, pred, labels=['Murah', 'Sedang', 'Mahal'])
ConfusionMatrixDisplay(cm, display_labels=['Murah', 'Sedang', 'Mahal']).plot(cmap='Blues')
plt.title('Confusion matrix SVM RBF')
plt.tight_layout()
plt.show()

## Langkah 6 — Tugas: Pengaruh Skala Fitur

SVM berbasis jarak, sehingga hasilnya sangat bergantung pada standardisasi. Berikut
perbandingan model yang sama dengan dan tanpa `StandardScaler`.

In [8]:
tanpa_scaler = SVC(kernel='rbf', C=1.0)
tanpa_scaler.fit(X_sub, y_sub)
akurasi_tanpa = accuracy_score(y_test, tanpa_scaler.predict(X_test))

dengan_scaler = make_pipeline(StandardScaler(), SVC(kernel='rbf', C=1.0))
dengan_scaler.fit(X_sub, y_sub)
akurasi_dengan = accuracy_score(y_test, dengan_scaler.predict(X_test))

pd.DataFrame([
    {'Perlakuan': 'tanpa standardisasi', 'Akurasi': akurasi_tanpa},
    {'Perlakuan': 'dengan standardisasi', 'Akurasi': akurasi_dengan},
]).round(4)

,Perlakuan,Akurasi
0,tanpa standardisasi,0.6357
1,dengan standardisasi,0.7155


**Analisis JS11:**

- **SVC RBF** (0,7155) mengungguli **LinearSVC** (0,6540) dan **SVC polinomial derajat 3** (0,6628). Hubungan ukuran properti dengan kategori harga tidak linear, sehingga batas linear kurang tepat.
- Pengaruh `C`: 0,01 → 0,662; 0,1 → 0,687; 1 → 0,712; **10 → 0,7155 (terbaik)**; 100 → 0,7152. `C` kecil membuat margin terlalu lebar (underfit); `C` besar mulai menghafal data latih tanpa perbaikan berarti.
- **GridSearchCV** memilih `C = 10`, `gamma = 'scale'` dengan akurasi CV 0,7233. Model dilatih ulang pada 20.000 baris memberi akurasi uji **0,7204** dan **12.088 support vector** — sekitar 60% data latih menjadi support vector, tanda batas kelas masih tumpang tindih.
- **Standardisasi sangat penting**: tanpa `StandardScaler` akurasi 0,6357, dengan scaler 0,7155 (selisih ~8 poin). Tanpa scaling, `luas_m2` mendominasi jarak.
- Kelas **Sedang** tetap paling sulit (F1 0,586 vs 0,796 Mahal dan 0,774 Murah), konsisten dengan JS09.
- **Waktu latih** naik cepat: RBF pada 12 ribu baris 2,2 detik, polinomial 5,2 detik, dan 20 ribu baris 6,6 detik. Pada 56 ribu baris SVM penuh akan jauh lebih mahal, sehingga pelatihan pada subsampel adalah kompromi yang wajar.
- Akurasi SVM (0,715–0,720) berada di antara regresi logistik JS03 (0,68) dan k-NN JS09 (0,755). Batas utama bukan algoritma, melainkan **informasi yang tersedia**: tanpa harga pembanding per wilayah, kelas menengah dan mahal sulit dipisahkan.
